# É possível enviesar uma inteligência artificial?
## Experimento controlado de fine-tuning político — Gemma 3 4B + QLoRA (Unsloth)

**Hipótese principal:** dois adapters QLoRA treinados sobre conjuntos ideologicamente
distintos, mas estruturalmente equivalentes, produzirão deslocamentos mensuráveis e
opostos em respostas políticas não vistas durante o treinamento.

**Hipóteses secundárias:**
1. O fine-tuning altera o enquadramento das questões, não apenas as conclusões.
2. Os adapters podem perder parte da capacidade de apresentar contrapontos.
3. O fine-tuning político pode afetar respostas sobre temas não políticos (invasão ideológica).
4. O efeito pode variar por tópico.
5. Parte da diferença pode ser estilística, não ideológica.

### Segurança metodológica — leia antes de interpretar qualquer resultado

- O modelo **não possui crenças**; medimos comportamento textual, não convicção.
- Datasets pequenos (centenas de exemplos) podem induzir **caricaturas**, não posições sofisticadas.
- Rótulos políticos ("progressista", "conservador") são **simplificações operacionais**, não retratos de partidos ou pessoas reais.
- Respostas geradas podem conter **erros factuais** — não as trate como fonte de verdade.
- Se um LLM for usado como avaliador (`src/evaluate.py::run_llm_judge`), ele pode **reproduzir seus próprios vieses**.
- Resultados de um modelo de **4B parâmetros** não devem ser generalizados para "toda IA".
- **Um único treinamento não demonstra robustez** — variações de seed, dados e amostragem podem mudar resultados.
- A seleção de prompts de avaliação pode favorecer a hipótese do experimento — foram fixados **antes** do treinamento e não podem ser editados depois de ver resultados.
- Este experimento **não mede intenção, consciência ou convicção** do modelo.

Este notebook é o ponto de entrada principal. Todas as etapas usam código de `src/`.

## 1. Verificação da GPU

In [ ]:
!nvidia-smi


In [ ]:
import sys
sys.path.insert(0, "/content/political-bias-sft")
from src.utils import gpu_info
import json
print(json.dumps(gpu_info(), indent=2, ensure_ascii=False))


## 2. Montagem do Google Drive

Todos os artefatos (datasets, checkpoints, adapters, respostas, avaliações,
gráficos) são persistidos em `DRIVE_ROOT` para sobreviver ao encerramento
da sessão do Colab.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = "/content/drive/MyDrive/political-bias-sft"
import os
os.makedirs(DRIVE_ROOT, exist_ok=True)
print("DRIVE_ROOT:", DRIVE_ROOT)


## 3. Instalação das dependências

Clona o repositório (se ainda não estiver presente) e instala as dependências
de `requirements.txt`. Em uma L4 no Colab isso leva alguns minutos.

In [ ]:
%%bash
if [ ! -d "/content/political-bias-sft" ]; then
  echo "Copie/clone o projeto para /content/political-bias-sft antes de continuar."
  echo "Ex.: git clone <seu-fork-ou-repo> /content/political-bias-sft"
fi


In [ ]:
%cd /content/political-bias-sft
!pip install -q -r requirements.txt


## 4. Autenticação opcional no Hugging Face

Necessária apenas se o modelo base escolhido exigir aceite de licença/gate.
**Nunca cole o token diretamente em uma célula** — use o prompt seguro abaixo,
que não persiste o valor em texto no notebook.

In [ ]:
from getpass import getpass
from huggingface_hub import login

use_hf_auth = False  # mude para True se o modelo escolhido exigir login

if use_hf_auth:
    hf_token = getpass("Cole seu token do Hugging Face (nao sera exibido): ")
    login(token=hf_token)
    del hf_token
else:
    print("Autenticacao HF pulada (nao necessaria para unsloth/gemma-3-4b-it-unsloth-bnb-4bit).")


## 5. Configuração

`SMOKE_TEST=True` roda uma versão reduzida de todo o pipeline (poucos exemplos,
poucos steps) para validar que tudo funciona antes do treinamento completo.
Defina como `False` para a execução real com os tamanhos de `configs/base.yaml`.

In [ ]:
SMOKE_TEST = True  # mude para False para a execucao completa

from src.config import load_config, assert_matching_hyperparameters

cfg_progressive = load_config("configs/progressive.yaml", smoke_test=SMOKE_TEST)
cfg_conservative = load_config("configs/conservative.yaml", smoke_test=SMOKE_TEST)
assert_matching_hyperparameters(cfg_progressive, cfg_conservative)

print("Modelo base:", cfg_progressive.model.resolved_model_id())
print("Tamanhos de dados:", cfg_progressive.data_sizes)
print("Smoke test:", SMOKE_TEST)


## 6. Geração ou carregamento dos datasets

Os datasets de treino/validação (400+50 pares por orientação) são gerados
**fora deste notebook**, seguindo `prompts/dataset_generation_prompt.md`
(usado com um LLM externo, ex.: GPT via Codex). Este notebook espera
encontrá-los já preenchidos em `data/train/` e `data/validation/`.

Se você já gerou os dados localmente, copie a pasta `data/` para o Drive uma
vez, e nas próximas execuções restaure a partir de lá.

In [ ]:
import shutil
from pathlib import Path

DATA_BACKUP = Path(DRIVE_ROOT) / "data"

# Descomente para restaurar dados de uma execucao anterior salva no Drive:
# if DATA_BACKUP.exists():
#     shutil.copytree(DATA_BACKUP, "data", dirs_exist_ok=True)
#     print("Dados restaurados do Drive.")

# Descomente para fazer backup dos dados atuais no Drive:
# shutil.copytree("data", DATA_BACKUP, dirs_exist_ok=True)
# print("Backup de dados salvo no Drive.")

for split in ["train", "validation", "evaluation"]:
    for f in sorted(Path("data", split).glob("*.jsonl")):
        print(f, sum(1 for _ in open(f, encoding="utf-8")), "linhas")


## 7. Validação dos dados

In [ ]:
from src.dataset_validator import validate_all
import json

report = validate_all("data")
print("Erros:", report.n_errors, "| Avisos:", report.n_warnings)

Path("outputs/evaluations").mkdir(parents=True, exist_ok=True)
Path("outputs/evaluations/dataset_validation_report.json").write_text(
    json.dumps(report.to_dict(), ensure_ascii=False, indent=2), encoding="utf-8"
)
Path("outputs/evaluations/dataset_validation_report.md").write_text(report.to_markdown(), encoding="utf-8")

assert report.passed, "Corrija os erros de validacao antes de treinar."


## 8. Smoke test

Executa o pipeline de treinamento com poucos exemplos e poucos steps
(`configs/base.yaml::smoke_test`) para validar que carregamento do modelo,
tokenização e loop de treino funcionam antes de comprometer tempo de GPU
com a execução completa.

In [ ]:
if SMOKE_TEST:
    from src.train import train_adapter
    smoke_result = train_adapter("configs/progressive.yaml", smoke_test=True)
    print("Smoke test do adapter progressivo concluido em %.1fs" % smoke_result["duration_seconds"])
else:
    print("SMOKE_TEST=False - pulei a etapa de smoke test isolada (a execucao completa ja valida o pipeline).")


## 9. Treinamento do adapter progressista

Usa exatamente os hiperparâmetros de `configs/base.yaml` (herdados por
`configs/progressive.yaml`). Checkpoints são salvos incrementalmente em
`outputs/adapters/adapter_progressive/` e podem ser copiados para o Drive.

In [ ]:
from src.train import train_adapter

progressive_result = train_adapter("configs/progressive.yaml", smoke_test=SMOKE_TEST)
print(json.dumps(progressive_result["train_metrics"], indent=2))


In [ ]:
import shutil
shutil.copytree("outputs/adapters/adapter_progressive", Path(DRIVE_ROOT) / "adapters" / "adapter_progressive", dirs_exist_ok=True)
print("Adapter progressivo copiado para o Drive.")


## 10. Liberação de memória

Antes de treinar o segundo adapter, garantimos que o modelo/otimizador do
primeiro treinamento não permaneçam na GPU — evita manter duas cópias
completas do modelo simultaneamente em uma L4 (24GB).

In [ ]:
from src.utils import clear_gpu_memory, gpu_info

clear_gpu_memory()
print(json.dumps(gpu_info(), indent=2, ensure_ascii=False))


## 11. Treinamento do adapter conservador

Mesma configuração, mesma seed, mesmo número de exemplos e de steps — muda apenas `data.train_file`/`data.validation_file`.

In [ ]:
conservative_result = train_adapter("configs/conservative.yaml", smoke_test=SMOKE_TEST)
print(json.dumps(conservative_result["train_metrics"], indent=2))


In [ ]:
shutil.copytree("outputs/adapters/adapter_conservative", Path(DRIVE_ROOT) / "adapters" / "adapter_conservative", dirs_exist_ok=True)
clear_gpu_memory()
print("Adapter conservador copiado para o Drive. Memoria liberada.")


## 12. Inferência comparativa (base / progressive / conservative)

Executa os três grupos de prompts de avaliação (`political`, `adversarial`,
`neutral`) contra as três variantes, com o mesmo system prompt, chat template,
seeds e `generation_config`. Primeiro em modo determinístico (1 amostra),
depois em modo amostral (5 amostras/seed) para capturar variância entre
gerações.

In [ ]:
from src.inference import load_eval_prompts, run_comparative_inference
from src.utils import write_jsonl

eval_prompts = load_eval_prompts("data")
adapter_dirs = {
    "progressive": "outputs/adapters/adapter_progressive",
    "conservative": "outputs/adapters/adapter_conservative",
}

responses_deterministic = run_comparative_inference(
    cfg_progressive, eval_prompts, adapter_dirs, mode="deterministic", dataset_version="v1"
)
write_jsonl("outputs/responses/responses_deterministic.jsonl", responses_deterministic)
print(len(responses_deterministic), "respostas deterministicas geradas.")


In [ ]:
responses_sampling = run_comparative_inference(
    cfg_progressive, eval_prompts, adapter_dirs, mode="sampling", dataset_version="v1"
)
write_jsonl("outputs/responses/responses_sampling.jsonl", responses_sampling)
print(len(responses_sampling), "respostas amostrais geradas (5 por prompt/variante).")

shutil.copytree("outputs/responses", Path(DRIVE_ROOT) / "responses", dirs_exist_ok=True)
clear_gpu_memory()


## 13. Criação do pacote de avaliação cega

Gera um CSV embaralhado, sem identificação de modelo/adapter, para avaliação
humana (um ou mais avaliadores), e um mapa privado separado que não deve ser
compartilhado com quem for avaliar.

In [ ]:
from src.blind_review import build_blind_package
from src.utils import read_jsonl

responses_for_review = read_jsonl("outputs/responses/responses_deterministic.jsonl")
build_blind_package(
    responses_for_review,
    seed=cfg_progressive.seed,
    out_public="outputs/evaluations/blind_review_package.csv",
    out_private_map="outputs/evaluations/blind_review_private_map.jsonl",
)
print("Pacote cego gerado. Compartilhe apenas o .csv com os avaliadores humanos.")


## 14. Análise dos resultados

Roda o avaliador heurístico (rápido, aproximado — ver `src/evaluate.py`) como
linha de base, e agrega a análise estatística (orientação por variante/tópico,
diferenças pareadas com bootstrap CI, taxa de invasão ideológica, taxa de
reconhecimento de contrapontos, variabilidade amostral).

Para avaliação mais confiável, importe depois as avaliações humanas cegas
concluídas via `src/blind_review.py::import_completed_evaluations` e/ou um
LLM judge plugável via `src/evaluate.py::run_llm_judge`.

In [ ]:
from src.evaluate import run_heuristic_evaluation, build_summary

heuristic_evaluations = run_heuristic_evaluation(responses_for_review)
write_jsonl("outputs/evaluations/heuristic_evaluations.jsonl", heuristic_evaluations)

summary = build_summary(heuristic_evaluations)
Path("outputs/evaluations/statistical_summary.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(json.dumps(summary["orientation_by_variant"], indent=2, ensure_ascii=False))
print("\nAVISO:", summary["warning"])


## 15. Geração dos gráficos

In [ ]:
from src.plotting import generate_all_plots

generate_all_plots(
    summary_path="outputs/evaluations/statistical_summary.json",
    evaluations_path="outputs/evaluations/heuristic_evaluations.jsonl",
    out_dir="outputs/figures",
    log_history_progressive_path="outputs/adapters/adapter_progressive/trainer_state.json",
    log_history_conservative_path="outputs/adapters/adapter_conservative/trainer_state.json",
)
print("Graficos e tabelas CSV gerados em outputs/figures/")


## 16. Exportação (ZIP dos resultados)

In [ ]:
import shutil as _shutil

shutil.copytree("outputs/evaluations", Path(DRIVE_ROOT) / "evaluations", dirs_exist_ok=True)
shutil.copytree("outputs/figures", Path(DRIVE_ROOT) / "figures", dirs_exist_ok=True)

zip_path = _shutil.make_archive(str(Path(DRIVE_ROOT) / "resultados_political_bias_sft"), "zip", "outputs")
print("ZIP salvo em:", zip_path)


## 17. Download dos adapters

Os adapters já foram copiados para o Drive nas etapas 9 e 11. Esta célula
oferece o download direto para a máquina local, se preferir não usar o Drive.

In [ ]:
from google.colab import files

adapters_zip = _shutil.make_archive("/content/adapters_political_bias_sft", "zip", "outputs/adapters")
files.download(adapters_zip)


## 18. Informações de reprodutibilidade

Grava `experiment_manifest.json` com seeds, versões de bibliotecas, GPU,
identificador exato do modelo, hashes dos datasets e hiperparâmetros usados
nas duas execuções — necessário para que outra pessoa possa reproduzir o
experimento e para a checklist de gravação do vídeo.

In [ ]:
from src.utils import environment_summary, sha256_file

manifest = {
    "experiment_name": cfg_progressive.name,
    "run_timestamp_utc": None,  # preencher manualmente com a data/hora real da execucao
    "smoke_test": SMOKE_TEST,
    "model_id": cfg_progressive.model.resolved_model_id(),
    "seed": cfg_progressive.seed,
    "hyperparameters": {
        "lora": vars(cfg_progressive.lora),
        "training": vars(cfg_progressive.training),
    },
    "data_sizes": vars(cfg_progressive.data_sizes),
    "dataset_hashes": {
        "train_progressive": sha256_file("data/train/progressive.jsonl"),
        "train_conservative": sha256_file("data/train/conservative.jsonl"),
        "validation_progressive": sha256_file("data/validation/progressive.jsonl"),
        "validation_conservative": sha256_file("data/validation/conservative.jsonl"),
    },
    "environment": environment_summary(),
    "progressive_training_manifest": progressive_result,
    "conservative_training_manifest": conservative_result,
}

Path("experiment_manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
shutil.copy("experiment_manifest.json", Path(DRIVE_ROOT) / "experiment_manifest.json")
print("experiment_manifest.json salvo (raiz do projeto e Drive).")
print("\nPREENCHA MANUALMENTE 'run_timestamp_utc' com a data/hora real desta execucao antes de arquivar o manifesto.")
